# 오늘의 학습 주제: Co-occurrence Matrix → PPMI → SVD로 Word Embedding 만들기

오늘은 Word2Vec으로 바로 가기 전에, 더 고전적이고 해석 가능한 방식으로 **단어 임베딩을 직접 만드는 과정**을 봅니다.

핵심 흐름은 다음입니다.

```text
corpus
→ co-occurrence matrix
→ PMI
→ PPMI
→ SVD
→ dense word embedding
```

---

## 1. 핵심 개념

### 1-1. Co-occurrence matrix

co-occurrence matrix는 단어들이 주변 문맥에서 얼마나 자주 같이 등장했는지를 세는 행렬입니다.

예를 들어 vocabulary가 다음과 같다고 합시다.

In [3]:
["cat", "dog", "eats", "fish", "car", "truck", "wheels"]

['cat', 'dog', 'eats', 'fish', 'car', 'truck', 'wheels']

그러면 co-occurrence matrix의 한 row는 특정 단어의 context profile입니다.

```text
cooc["cat"] = cat 주변에 어떤 단어들이 얼마나 자주 나왔는가
```

즉, 단어 자체를 직접 의미 벡터로 만드는 것이 아니라,

```text
단어의 의미 ≈ 그 단어가 등장하는 문맥의 분포
```

라는 가정을 사용합니다.

이것이 distributional hypothesis입니다.

> 비슷한 문맥에 등장하는 단어는 비슷한 의미를 가진다.

---

## 2. 왜 raw count만으로는 부족한가?

co-occurrence count만 쓰면 자주 등장하는 단어가 과도하게 중요해집니다.

예를 들어:

```text
the
is
of
and
in
```

같은 단어들은 거의 모든 단어 주변에 등장합니다.

그래서 raw count만 보면, 의미적으로 중요한 관계보다 단순 빈도 효과가 커질 수 있습니다.

예를 들어 `cat` 주변에 `the`가 100번 나오고, `milk`가 5번 나왔다고 해서 `the`가 `cat`의 의미를 더 잘 설명한다고 보기는 어렵습니다.

그래서 단순 count 대신 **PMI**를 사용합니다.

---

## 3. PMI: Pointwise Mutual Information

PMI는 두 단어가 독립적으로 등장하는 경우에 비해 실제로 얼마나 더 자주 같이 등장하는지를 측정합니다.

수식은 다음입니다.

```text
PMI(w, c) = log( P(w, c) / (P(w) P(c)) )
```

의미는 다음과 같습니다.

```text
P(w, c): 단어 w와 context c가 함께 등장할 확률
P(w): 단어 w가 등장할 확률
P(c): context c가 등장할 확률
```

PMI가 크다는 것은 다음을 의미합니다.

```text
w와 c가 우연히 같이 등장한 것보다
실제로 훨씬 강하게 연결되어 있다.
```

예를 들어:

```text
PMI(cat, milk)   → 높을 수 있음
PMI(car, wheels) → 높을 수 있음
PMI(cat, the)    → 낮을 수 있음
```

---

## 4. PPMI: Positive PMI

PMI는 음수가 될 수 있습니다.

```text
PMI(w, c) < 0
```

이것은 두 단어가 기대보다 덜 같이 등장한다는 뜻입니다.

하지만 임베딩을 만들 때는 보통 음수 PMI를 0으로 잘라냅니다.

```text
PPMI(w, c) = max(PMI(w, c), 0)
```

즉:

```text
양의 연관성만 남긴다.
```

이렇게 만든 행렬을 **PPMI matrix**라고 합니다.

---

## 5. 왜 SVD를 쓰는가?

PPMI matrix는 보통 크고 sparse합니다.

예를 들어 vocabulary 크기가 50,000이면 행렬 크기는 다음과 같습니다.

```text
50,000 x 50,000
```

이 행렬의 각 row는 단어 벡터로 볼 수 있지만, 너무 큽니다.

그래서 SVD를 사용해 저차원 dense vector로 압축합니다.

```text
PPMI matrix
→ SVD
→ 50차원, 100차원, 300차원 word embedding
```

SVD는 대략 다음 분해를 합니다.

```text
M ≈ U S Vᵀ
```

여기서 `U`의 앞쪽 k개 차원을 사용하면 단어 임베딩을 얻을 수 있습니다.

embedding = U[:, :k] * S[:k]

---

# 오늘의 작은 실습

아래 코드는 작은 corpus에서 co-occurrence matrix, PPMI matrix, SVD embedding을 직접 만듭니다.

In [4]:
import numpy as np

corpus = [
    "cat eats fish",
    "dog eats meat",
    "cat drinks milk",
    "dog drinks water",
    "car uses fuel",
    "truck uses diesel",
    "car has wheels",
    "truck has wheels",
]

tokenized = [sent.split() for sent in corpus]

vocab = sorted(set(word for sent in tokenized for word in sent))
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

print(vocab)

['car', 'cat', 'diesel', 'dog', 'drinks', 'eats', 'fish', 'fuel', 'has', 'meat', 'milk', 'truck', 'uses', 'water', 'wheels']


---

## Step 1. Co-occurrence matrix 만들기

In [5]:
def build_cooc_matrix(tokenized, word2idx, window_size=2):
    V = len(word2idx)
    cooc = np.zeros((V, V), dtype=np.float64)

    for sent in tokenized:
        ids = [word2idx[w] for w in sent]

        for center_pos, center_id in enumerate(ids):
            left = max(0, center_pos - window_size)
            right = min(len(ids), center_pos + window_size + 1)

            for context_pos in range(left, right):
                if context_pos == center_pos:
                    continue

                context_id = ids[context_pos]
                cooc[center_id, context_id] += 1.0

    return cooc

cooc = build_cooc_matrix(tokenized, word2idx, window_size=2)

print(cooc)

[[0. 0. 0. 0. 0. 0. 0. 1. 1. 0. 0. 0. 1. 0. 1.]
 [0. 0. 0. 0. 1. 1. 1. 0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 1. 0. 0.]
 [0. 0. 0. 0. 1. 1. 0. 0. 0. 1. 0. 0. 0. 1. 0.]
 [0. 1. 0. 1. 0. 0. 0. 0. 0. 0. 1. 0. 0. 1. 0.]
 [0. 1. 0. 1. 0. 0. 1. 0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 2.]
 [0. 0. 0. 1. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 1. 0. 1.]
 [1. 0. 1. 0. 0. 0. 0. 1. 0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0. 0. 2. 0. 0. 1. 0. 0. 0.]]


---

## Step 2. PPMI matrix 만들기

In [6]:
def build_ppmi_matrix(cooc, eps=1e-8):
    total = np.sum(cooc)

    p_wc = cooc / total
    p_w = np.sum(cooc, axis=1, keepdims=True) / total
    p_c = np.sum(cooc, axis=0, keepdims=True) / total

    pmi = np.log((p_wc + eps) / (p_w @ p_c + eps))
    ppmi = np.maximum(pmi, 0.0)

    return ppmi

ppmi = build_ppmi_matrix(cooc)

np.set_printoptions(precision=3, suppress=True)
print(ppmi)

[[0.    0.    0.    0.    0.    0.    0.    1.792 1.099 0.    0.    0.
  1.099 0.    1.099]
 [0.    0.    0.    0.    1.099 1.099 1.792 0.    0.    0.    1.792 0.
  0.    0.    0.   ]
 [0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    1.792
  1.792 0.    0.   ]
 [0.    0.    0.    0.    1.099 1.099 0.    0.    0.    1.792 0.    0.
  0.    1.792 0.   ]
 [0.    1.099 0.    1.099 0.    0.    0.    0.    0.    0.    1.792 0.
  0.    1.792 0.   ]
 [0.    1.099 0.    1.099 0.    0.    1.792 0.    0.    1.792 0.    0.
  0.    0.    0.   ]
 [0.    1.792 0.    0.    0.    1.792 0.    0.    0.    0.    0.    0.
  0.    0.    0.   ]
 [1.792 0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
  1.792 0.    0.   ]
 [1.099 0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    1.099
  0.    0.    1.792]
 [0.    0.    0.    1.792 0.    1.792 0.    0.    0.    0.    0.    0.
  0.    0.    0.   ]
 [0.    1.792 0.    0.    1.792 0.    0.    0.    0.    0.    0.    0.
  0

---

## Step 3. SVD로 dense embedding 만들기

In [12]:
U, S, VT = np.linalg.svd(ppmi)

embedding_dim = 2
embeddings = U[:, :embedding_dim] * S[:embedding_dim]

print("embedding shape:", embeddings.shape)
print(embeddings)

embedding shape: (15, 2)
[[-0.     1.847]
 [-1.949 -0.   ]
 [ 0.     1.569]
 [-1.949 -0.   ]
 [-1.949 -0.   ]
 [-1.949 -0.   ]
 [-1.441 -0.   ]
 [ 0.     1.569]
 [ 0.     1.475]
 [-1.441 -0.   ]
 [-1.441 -0.   ]
 [-0.     1.847]
 [-0.     2.131]
 [-1.441 -0.   ]
 [ 0.     1.475]]


---

## Step 4. 가장 가까운 단어 찾기

In [8]:
def cosine(a, b, eps=1e-8):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + eps)

def nearest(word, embeddings, top_k=5):
    wid = word2idx[word]
    query = embeddings[wid]

    scores = []
    for i in range(len(vocab)):
        if i == wid:
            continue
        score = cosine(query, embeddings[i])
        scores.append((idx2word[i], score))

    return sorted(scores, key=lambda x: x[1], reverse=True)[:top_k]

for word in ["cat", "dog", "car", "truck"]:
    print("\nQUERY:", word)
    print(nearest(word, embeddings))


QUERY: cat
[('dog', np.float64(0.9999999973663731)), ('drinks', np.float64(0.9999999973663731)), ('eats', np.float64(0.9999999973663731)), ('fish', np.float64(0.9999999964379873)), ('meat', np.float64(0.9999999964379873))]

QUERY: dog
[('cat', np.float64(0.9999999973663731)), ('drinks', np.float64(0.9999999973663731)), ('eats', np.float64(0.9999999973663731)), ('fish', np.float64(0.9999999964379873)), ('meat', np.float64(0.9999999964379873))]

QUERY: car
[('uses', np.float64(0.9999999974597812)), ('truck', np.float64(0.9999999970700008)), ('diesel', np.float64(0.9999999965498924)), ('fuel', np.float64(0.9999999965498924)), ('has', np.float64(0.9999999963307269))]

QUERY: truck
[('uses', np.float64(0.9999999974597812)), ('car', np.float64(0.9999999970700008)), ('diesel', np.float64(0.9999999965498924)), ('fuel', np.float64(0.9999999965498924)), ('has', np.float64(0.9999999963307269))]


---

# 관찰 포인트

실행한 뒤 다음을 확인하세요.

```text
cat과 dog가 가까운가?
car와 truck이 가까운가?
fish, meat, milk, water는 동물 관련 단어 쪽에 가까운가?
fuel, diesel, wheels는 차량 관련 단어 쪽에 가까운가?
```

데이터가 매우 작기 때문에 결과가 완벽하지 않을 수 있습니다.

중요한 것은 결과 자체보다 과정입니다.

```text
같이 등장한 빈도
→ 우연 이상의 연관성
→ 저차원 dense vector
```

이 흐름이 오늘의 핵심입니다.

---

# 오늘의 핵심 정리

오늘 배운 방식은 neural network를 쓰지 않습니다.

하지만 임베딩의 핵심 아이디어를 잘 보여줍니다.

```text
단어의 의미는 그 단어가 등장하는 문맥 분포로 표현할 수 있다.
```

그리고 다음 변환을 거칩니다.

```text
raw count
→ PMI
→ PPMI
→ SVD
→ dense embedding
```

이 방식은 Word2Vec이나 GloVe를 이해하기 위한 좋은 중간 단계입니다.

특히 GloVe는 global co-occurrence statistics를 사용한다는 점에서 오늘 내용과 연결됩니다.

---

# 복습 질문

1. co-occurrence matrix의 row 하나는 무엇을 의미하는가?

2. raw count만 사용하면 왜 자주 등장하는 단어가 문제를 일으킬 수 있는가?

3. PMI는 두 단어의 어떤 관계를 측정하는가?

4. PPMI에서 음수 PMI를 0으로 자르는 이유는 무엇인가?

5. SVD를 사용하면 sparse한 PPMI matrix가 어떤 형태로 바뀌는가?

---

# 추가 실습

corpus에 다음 문장을 추가해보세요.

In [9]:
extra_sentences = [
    "kitten drinks milk",
    "puppy eats meat",
    "bus has wheels",
    "train uses electricity",
]

그다음 다시 실행해서 비교하세요.

In [10]:
for word in ["cat", "dog", "car", "truck", "kitten", "bus"]:
    print("\nQUERY:", word)
    print(nearest(word, embeddings))


QUERY: cat
[('dog', np.float64(0.9999999973663731)), ('drinks', np.float64(0.9999999973663731)), ('eats', np.float64(0.9999999973663731)), ('fish', np.float64(0.9999999964379873)), ('meat', np.float64(0.9999999964379873))]

QUERY: dog
[('cat', np.float64(0.9999999973663731)), ('drinks', np.float64(0.9999999973663731)), ('eats', np.float64(0.9999999973663731)), ('fish', np.float64(0.9999999964379873)), ('meat', np.float64(0.9999999964379873))]

QUERY: car
[('uses', np.float64(0.9999999974597812)), ('truck', np.float64(0.9999999970700008)), ('diesel', np.float64(0.9999999965498924)), ('fuel', np.float64(0.9999999965498924)), ('has', np.float64(0.9999999963307269))]

QUERY: truck
[('uses', np.float64(0.9999999974597812)), ('car', np.float64(0.9999999970700008)), ('diesel', np.float64(0.9999999965498924)), ('fuel', np.float64(0.9999999965498924)), ('has', np.float64(0.9999999963307269))]

QUERY: kitten


KeyError: 'kitten'

확인할 점은 다음입니다.

```text
kitten이 cat 쪽에 가까워지는가?
puppy가 dog 쪽에 가까워지는가?
bus가 car/truck 쪽에 가까워지는가?
train이 vehicle 계열 단어와 가까워지는가?
```

다음 학습에서는 이 흐름을 neural objective로 바꿔서 **Skip-gram with Negative Sampling**을 직접 구현하는 단계로 넘어가면 됩니다.